# 🏥 ViBioMIR Interactive Search & Offline Testing Notebook
Notebook này phục vụ các mục tiêu:
1. **Đồng bộ Index từ Hugging Face Bucket** (`nieths/ViBioMIR`) - Lựa chọn SQ8 FAISS, BM25s hoặc Hybrid.
2. **Thử nghiệm tìm kiếm tương tác (Interactive Query Search)**: Nhập query bất kỳ (Việt / Anh / Trung) và xem kết quả trực tiếp của Dense FAISS, BM25, RRF Fusion và BGE-Reranker-v2-m3.
3. **Kiểm tra & So sánh chiến lược trích xuất (Chunk Strategy Evaluation)**: So sánh giữa nộp quá nhiều chunks (120 chunks / 60 docs) vs Chiến lược chuẩn Top 1 (1 chunk / doc) theo $F_2$-Macro.
4. **Sinh file nộp bài chính thức (`submission.zip`)** tốc độ cao.

In [ ]:
# 1. Cài đặt các thư viện cần thiết siêu tốc với uv
!pip install -q uv
!uv pip install --system --no-cache faiss-gpu pyarrow "huggingface_hub[cli]" bm25s rank-bm25 pyvi jieba loguru pydantic pyyaml tqdm flagembedding sentence-transformers

In [ ]:
# 2. Clone hoặc Cập nhật Codebase từ GitHub
import os
import sys
from pathlib import Path

if Path("/kaggle/working").exists():
    PROJECT_DIR = Path("/kaggle/working/med-doc-retrieval")
    if not PROJECT_DIR.exists():
        !git clone https://github.com/neiths/med-doc-retrieval.git $PROJECT_DIR
    %cd $PROJECT_DIR
    !git pull origin main
    sys.path.insert(0, str(PROJECT_DIR))
else:
    PROJECT_DIR = Path.cwd()
    sys.path.insert(0, str(PROJECT_DIR))

print("Working directory:", os.getcwd())

In [ ]:
# 3. Tải Index từ Hugging Face Bucket (SQ8 FAISS + Slim Metadata + BM25s)
import os
from pathlib import Path
from huggingface_hub import HfApi

HF_TOKEN = os.environ.get("HF_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

indices_dir = Path("data/indices")
indices_dir.mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)

# Danh sách file tối ưu cho Hybrid Retrieval (~5.9 GB total)
sync_files = [
    ("indices/dense_index_sq8.faiss", "data/indices/dense_index_sq8.faiss"),
    ("indices/chunks_meta_slim.sqlite", "data/indices/chunks_meta_slim.sqlite"),
    ("indices/dense_metadata_sq8.json", "data/indices/dense_metadata_sq8.json"),
    ("processed/queries_enriched.jsonl", "data/processed/queries_enriched.jsonl"),
]

# Kiểm tra file nào còn thiếu
missing = [(rem, loc) for rem, loc in sync_files if not Path(loc).exists() or Path(loc).stat().st_size == 0]

if missing:
    print(f"📥 Đang tải {len(missing)} file index cần thiết từ HF Bucket...")
    api = HfApi(token=HF_TOKEN or None)
    api.download_bucket_files(bucket_id="nieths/ViBioMIR", files=missing, token=HF_TOKEN or None)
    print("✅ Tải index hoàn tất!")
else:
    print("✅ Toàn bộ file index đã có sẵn!")

!ls -lh data/indices

In [ ]:
# 4. Khởi tạo Pipeline & Load Index
from src.config import load_config
from src.pipeline import MedicalRetrievalPipeline

config = load_config("configs/config.yaml")
config.pubmed.source_api = "none"  # Tắt live HTTP call để đạt tốc độ tối đa

pipeline = MedicalRetrievalPipeline(config=config)
pipeline.load_indices(
    indices_dir="data/indices",
    dense_dir="data/indices",
    chunks_meta_path="data/indices/chunks_meta_slim.sqlite",
)
print("🚀 Pipeline đã sẵn sàng!")

In [ ]:
# 5. Thử nghiệm Truy vấn Tương tác (Interactive Test)
test_query = "Bệnh nhân bị xuất huyết giảm tiểu cầu vô căn điều trị bằng thuốc gì đầu tay?"

print(f"🔍 Đang tìm kiếm cho query: '{test_query}'")
result = pipeline.search_query(test_query)

print("\n--- 🏆 TOP TÀI LIỆU (DOC_IDS) ---")
for rank, did in enumerate(result["relevant_docs"][:5], 1):
    print(f"#{rank} Doc ID: {did}")

print("\n--- 📑 TOP CHUNKS BẰNG CHỨNG TRÍCH XUẤT ---")
for rank, ch in enumerate(result["relevant_chunks"][:3], 1):
    snippet = ch['chunk_text'][:200].replace('\n', ' ')
    print(f"#{rank} Doc ID {ch['doc_id']}:\n   \"{snippet}...\"\n")

In [ ]:
# 6. Chạy thử nghiệm trên 5 query ngẫu nhiên trong tập test
import json
import pyarrow.parquet as pq

queries_path = Path("data/processed/queries_enriched.jsonl")
sample_queries = []

if queries_path.exists():
    with open(queries_path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 5: break
            sample_queries.append(json.loads(line))
else:
    # Fallback to query.parquet
    table = pq.read_table("data/raw/vibio_mir/query.parquet")
    sample_queries = table.to_pylist()[:5]

print(f"Chạy thử nghiệm trên {len(sample_queries)} queries mẫu:")
for q in sample_queries:
    qid = q["id"]
    q_text = q.get("query") or q.get("original_query")
    res = pipeline.search_query(q_text)
    print(f"\n[Query #{qid}]: {q_text[:70]}...")
    print(f"  -> Top Docs: {res['relevant_docs'][:4]}")
    print(f"  -> Số Chunks: {len(res['relevant_chunks'])}")

In [ ]:
# 7. Chạy Inference toàn bộ tập 1,200 câu hỏi với CHIẾN LƯỢC TOP 1 (Tối ưu điểm Macro F2)
# Lưu ý: Áp dụng top_k_docs=4, top_k_chunks=4, max_chunks_per_doc=1 để nâng Chunks Precision lên 0.25+
!HF_HOME=/tmp/hf python scripts/run_inference_only.py \
    --indices-dir "data/indices" \
    --chunks-meta-path "data/indices/chunks_meta_slim.sqlite" \
    --hybrid-top-k 200 \
    --top-k-docs 4 \
    --top-k-chunks 4 \
    --max-chunks-per-doc 1 \
    --batch-size 16 \
    --disable-pubmed-network \
    --output-name submission_top1_strategy.json

In [ ]:
# 8. Tải File Submission về máy bằng HTML Link
import base64
from IPython.display import HTML, display

zip_candidates = list(Path("/kaggle/working").glob("*.zip")) if Path("/kaggle/working").exists() else list(Path("outputs/submissions").glob("*.zip"))

if zip_candidates:
    latest_zip = sorted(zip_candidates, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    with open(latest_zip, "rb") as f:
        data_b64 = base64.b64encode(f.read()).decode()
    
    html_button = f'''
    <div style="padding:15px; background:#e8f4fd; border:2px solid #2196f3; border-radius:8px;">
       <h3 style="color:#0b5394; margin-top:0;">🎉 Submission đã sẵn sàng!</h3>
       <p>File: <b>{latest_zip.name}</b> ({latest_zip.stat().st_size / (1024*1024):.2f} MB)</p>
       <a href="data:application/zip;base64,{data_b64}" download="{latest_zip.name}" 
          style="display:inline-block; padding:12px 24px; background:#2196f3; color:white; 
                 text-decoration:none; border-radius:6px; font-weight:bold; font-size:15px;">
          ⬇ BẤM VÀO ĐÂY ĐỂ TẢI {latest_zip.name} VỀ MÁY
       </a>
    </div>
    '''
    display(HTML(html_button))
else:
    print("Chưa tìm thấy file submission .zip nào. Hãy chạy cell 7 trước!")